# Bölüm 9: Çoklu ve Lojistik Regresyon

**Haydar Kılıç — Yapay Zeka Mühendisliği**

Bu notebook, Bölüm 9'daki teorik konuların Python ile uygulamalı gösterimidir.

## İçindekiler
1. [Çoklu Doğrusal Regresyon](#1)
2. [Kategorik Değişkenler ve Referans Düzeyi](#2)
3. [Regresyon Katsayılarının Yorumu](#3)
4. [R² ve Düzeltilmiş R²](#4)
5. [Model Seçimi (Geriye Doğru Eleme)](#5)
6. [Model Koşullarının Kontrolü](#6)
7. [Lojistik Regresyon ve GLM](#7)
8. [Donner Partisi: Hayatta Kalma Analizi](#8)
9. [Kuş Beslemek ve Akciğer Kanseri](#9)
10. [ROC Eğrisi ve Spam Filtresi](#10)

In [ ]:
# ── Gerekli Kütüphaneler ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Grafik stili
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')

print('Kütüphaneler başarıyla yüklendi ✓')

---
<a id='1'></a>
## 1. Çoklu Doğrusal Regresyon

**Basit doğrusal regresyon:** $\hat{y} = \beta_0 + \beta_1 x$

**Çoklu doğrusal regresyon:** $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$

Sunumdaki **kitap ağırlıkları** örneği: Kitabın ağırlığını *hacim* ve *kapak türü* ile tahmin edeceğiz.

In [ ]:
# ── Kitap Ağırlıkları Verisi (Sunumdaki Tablo) ───────────────────────────────
kitap_data = {
    'agirlik':  [800, 950, 1050, 350,  750,  600, 1075, 250, 700, 650, 975, 350, 950, 425, 725],
    'hacim':    [885, 1016, 1125, 239, 701,  641, 1228, 412, 953, 929, 1492, 419, 1010, 595, 1034],
    'kapak':    ['sert','sert','sert','sert','sert','sert','sert',
                 'yumusak','yumusak','yumusak','yumusak','yumusak','yumusak','yumusak','yumusak']
}
kitap = pd.DataFrame(kitap_data)
kitap['kapak_kodu'] = (kitap['kapak'] == 'yumusak').astype(int)  # 0=sert, 1=yumuşak

print(kitap.to_string(index=False))

In [ ]:
# ── 1a. Sadece Hacim ile Basit Regresyon ─────────────────────────────────────
model_basit = smf.ols('agirlik ~ hacim', data=kitap).fit()
print('=== MODEL 1: Sadece Hacim ===')
print(model_basit.summary().tables[1])
print(f'R² = {model_basit.rsquared:.4f}   Düzeltilmiş R² = {model_basit.rsquared_adj:.4f}')

In [ ]:
# ── 1b. Hacim + Kapak Türü ile Çoklu Regresyon ───────────────────────────────
model_coklu = smf.ols('agirlik ~ hacim + kapak_kodu', data=kitap).fit()
print('=== MODEL 2: Hacim + Kapak Türü ===')
print(model_coklu.summary().tables[1])
print(f'R² = {model_coklu.rsquared:.4f}   Düzeltilmiş R² = {model_coklu.rsquared_adj:.4f}')

**Doğrusal model denklemleri:**

$$\hat{\text{ağırlık}} = 197.96 + 0.72 \times \text{hacim} - 184.05 \times \text{kapak\_yumuşak}$$

- **Sert kapak** (`kapak_kodu = 0`): $\hat{y} = 197.96 + 0.72 \times \text{hacim}$
- **Yumuşak kapak** (`kapak_kodu = 1`): $\hat{y} = 13.91 + 0.72 \times \text{hacim}$

In [ ]:
# ── 1c. İki Regresyon Doğrusunu Görselleştirme ───────────────────────────────
b0 = model_coklu.params['Intercept']
b1 = model_coklu.params['hacim']
b_kapak = model_coklu.params['kapak_kodu']

h_aralik = np.linspace(200, 1550, 200)
y_sert     = b0 + b1 * h_aralik
y_yumusak  = b0 + b_kapak + b1 * h_aralik

fig, ax = plt.subplots(figsize=(8, 5))

sert_mask = kitap['kapak'] == 'sert'
ax.scatter(kitap.loc[sert_mask, 'hacim'],    kitap.loc[sert_mask, 'agirlik'],
           marker='s', color='steelblue', s=70, label='Sert kapak', zorder=3)
ax.scatter(kitap.loc[~sert_mask, 'hacim'],   kitap.loc[~sert_mask, 'agirlik'],
           marker='o', color='tomato',     s=70, label='Yumuşak kapak', zorder=3)

ax.plot(h_aralik, y_sert,    color='steelblue', lw=2, label=f'Sert:    ŷ = {b0:.1f} + {b1:.2f}×hacim')
ax.plot(h_aralik, y_yumusak, color='tomato',    lw=2, linestyle='--',
        label=f'Yumuşak: ŷ = {b0+b_kapak:.1f} + {b1:.2f}×hacim')

ax.set_xlabel('Hacim (cm³)')
ax.set_ylabel('Ağırlık (g)')
ax.set_title('Kitap Ağırlıkları – Çoklu Regresyon Modeli')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nKatsayı Yorumu:')
print(f'  Hacim eğimi     : Kapak türü sabit iken, hacim 1 cm³ arttığında ağırlık ~{b1:.2f} g artar.')
print(f'  Kapak eğimi     : Hacim sabit iken, yumuşak kapaklılar sert kapaklılardan ~{-b_kapak:.0f} g daha hafiftir.')
print(f'  Kesim noktası   : Hacmi 0 olan sert kapaklı kitabın tahmini ağırlığı {b0:.1f} g (anlamsız ama eğriyi konumlandırır).')

In [ ]:
# ── 1d. Örnek Tahmin ─────────────────────────────────────────────────────────
# Sunumdaki soru: 600 cm³ hacminde yumuşak kapaklı kitabın tahmini ağırlığı?
hacim_ornek = 600
tahmin_yumusak = b0 + b_kapak + b1 * hacim_ornek
tahmin_sert    = b0             + b1 * hacim_ornek

print(f'600 cm³ hacimli kitap tahminleri:')
print(f'  Yumuşak kapak : {b0:.2f} + {b1:.2f}×{hacim_ornek} + ({b_kapak:.2f})×1 = {tahmin_yumusak:.2f} g')
print(f'  Sert kapak    : {b0:.2f} + {b1:.2f}×{hacim_ornek} + ({b_kapak:.2f})×0 = {tahmin_sert:.2f} g')

---
<a id='2'></a>
## 2. Kategorik Değişkenler ve Referans Düzeyi

Kategorik değişkenler modele **kukla (dummy) değişken** olarak eklenir.
- $k$ düzeyli bir kategorik değişken için $k-1$ kukla değişken oluşturulur.
- Modele girmeyen düzey **referans düzeyi**dir; katsayısı sıfır kabul edilir.

Sunumdaki örnek: Yoksulluk oranını 4 bölgeyle (kuzeydoğu, orta batı, batı, güney) modellemek.

In [ ]:
# ── Yoksulluk – Bölge Örneği (Sunumdaki Katsayılar) ─────────────────────────
# Sunumdan: referans = kuzeydoğu, kesim = 9.50
np.random.seed(42)
bolge_cats = ['kuzeydoğu', 'ortabatı', 'batı', 'güney']
bolge_params = {'kuzeydoğu': 0, 'ortabatı': 0.03, 'batı': 1.79, 'güney': 4.16}
kesim = 9.50

n = 51
bolgeler = np.random.choice(bolge_cats, size=n)
yoksulluk = [kesim + bolge_params[b] + np.random.normal(0, 1.5) for b in bolgeler]

df_bolge = pd.DataFrame({'bolge': bolgeler, 'yoksulluk': yoksulluk})

# Referans düzeyi = kuzeydoğu (CategoricalDtype ile sırala)
df_bolge['bolge'] = pd.Categorical(df_bolge['bolge'],
                                    categories=bolge_cats)

model_bolge = smf.ols('yoksulluk ~ C(bolge, Treatment("kuzeydoğu"))', data=df_bolge).fit()
print(model_bolge.summary().tables[1])
print('\n→ Referans düzeyi: kuzeydoğu (kesim noktasında gizlidir)')

In [ ]:
# ── Görselleştirme: Bölgelere göre Yoksulluk Kutu Grafiği ───────────────────
fig, ax = plt.subplots(figsize=(7, 4))
df_bolge.boxplot(column='yoksulluk', by='bolge', ax=ax,
                 boxprops=dict(color='steelblue'),
                 medianprops=dict(color='tomato', lw=2))
ax.set_title('Bölgelere Göre Yoksulluk Oranı')
ax.set_xlabel('Bölge')
ax.set_ylabel('Yoksulluk Oranı (%)')
plt.suptitle('')
plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. Çoklu Doğrusal Bağlantı (Multicollinearity)

> Açıklayıcı değişkenler birbiriyle yüksek korelasyon gösterdiğinde **çoklu doğrusal bağlantı** ortaya çıkar. Bu durum katsayı tahminlerini dengesizleştirir ve yorumlamayı güçleştirir.

In [ ]:
# ── Yoksulluk – Kadın Hane Reisi + Beyaz Örneği ──────────────────────────────
np.random.seed(7)
n = 51
kadin_hane = np.random.uniform(8, 18, n)
# 'beyaz' ile kadin_hane arasında negatif korelasyon oluşturalım (r ≈ -0.75)
beyaz = 95 - 2.0 * kadin_hane + np.random.normal(0, 8, n)
beyaz = np.clip(beyaz, 30, 100)
yoksulluk = 3.31 + 0.69 * kadin_hane + np.random.normal(0, 1.8, n)

df_yok = pd.DataFrame({'yoksulluk': yoksulluk,
                        'kadin_hane': kadin_hane,
                        'beyaz': beyaz})

m1 = smf.ols('yoksulluk ~ kadin_hane', data=df_yok).fit()
m2 = smf.ols('yoksulluk ~ kadin_hane + beyaz', data=df_yok).fit()

print('Model 1 – Sadece kadın hane:')
print(m1.summary().tables[1])
print(f'  R² = {m1.rsquared:.3f}  |  Düz. R² = {m1.rsquared_adj:.3f}\n')

print('Model 2 – Kadın hane + Beyaz:')
print(m2.summary().tables[1])
print(f'  R² = {m2.rsquared:.3f}  |  Düz. R² = {m2.rsquared_adj:.3f}')

In [ ]:
# ── VIF (Varyans Şişirme Faktörü) ile Çoklu Doğrusal Bağlantı Tespiti ────────
# VIF > 5-10 → çoklu doğrusal bağlantı var
X_vif = sm.add_constant(df_yok[['kadin_hane', 'beyaz']])
vif_data = pd.DataFrame({
    'Değişken': ['kadin_hane', 'beyaz'],
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(2)]
})
print('Varyans Şişirme Faktörleri (VIF):')
print(vif_data.to_string(index=False))
print('\nKorelasyon matrisi:')
print(df_yok[['kadin_hane', 'beyaz']].corr().round(3))
print('\n→ VIF yüksekse değişken çiftinden birini modelden çıkarmayı düşünün.')

---
<a id='4'></a>
## 4. R² ve Düzeltilmiş R²

$$R^2 = \frac{SS_{Model}}{SS_{Toplam}} = 1 - \frac{SS_{Hata}}{SS_{Toplam}}$$

$$R^2_{\text{düz}} = 1 - \frac{SS_{Hata}}{SS_{Toplam}} \times \frac{n-1}{n-p-1}$$

> **Neden Düzeltilmiş R²?** Modele anlamsız değişken eklendiğinde R² artar ama Düz. R² artmaz (ceza uygular).

In [ ]:
# ── R² ve Düzeltilmiş R² Hesabı (Manuel) ────────────────────────────────────
y = df_yok['yoksulluk'].values
y_hat1 = m1.fittedvalues.values
y_hat2 = m2.fittedvalues.values

SS_toplam = np.sum((y - y.mean())**2)
SS_hata1  = np.sum((y - y_hat1)**2)
SS_hata2  = np.sum((y - y_hat2)**2)
n_obs = len(y)

def r2_duz(SS_hata, SS_toplam, n, p):
    return 1 - (SS_hata / SS_toplam) * ((n - 1) / (n - p - 1))

print('=== ANOVA Tarzı Hesaplama ===')
print(f'SS_Toplam = {SS_toplam:.2f}')
print(f'SS_Hata (Model 1, p=1) = {SS_hata1:.2f}')
print(f'SS_Hata (Model 2, p=2) = {SS_hata2:.2f}')
print()
print(f'Model 1  R² = {1 - SS_hata1/SS_toplam:.3f}')
print(f'Model 1  Düz. R² = {r2_duz(SS_hata1, SS_toplam, n_obs, 1):.3f}')
print(f'Model 2  R² = {1 - SS_hata2/SS_toplam:.3f}')
print(f'Model 2  Düz. R² = {r2_duz(SS_hata2, SS_toplam, n_obs, 2):.3f}')
print()
print('→ R² artmasına rağmen Düzeltilmiş R² neredeyse aynı → beyaz değişkeni ek bilgi sağlamıyor.')

In [ ]:
# ── Görselleştirme: R² vs Düzeltilmiş R² ────────────────────────────────────
# Modele rastgele değişkenler ekleyerek R² nasıl değişiyor?
np.random.seed(42)
r2_list, r2_adj_list = [], []
max_p = 15

for p in range(1, max_p + 1):
    X_ekstra = np.random.randn(n_obs, p)
    X_all = np.column_stack([df_yok['kadin_hane'].values, X_ekstra])
    X_sm = sm.add_constant(X_all)
    m_tmp = sm.OLS(y, X_sm).fit()
    r2_list.append(m_tmp.rsquared)
    r2_adj_list.append(m_tmp.rsquared_adj)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, max_p+1), r2_list,     'o-', color='steelblue', label='R²')
ax.plot(range(1, max_p+1), r2_adj_list, 's--', color='tomato',    label='Düzeltilmiş R²')
ax.set_xlabel('Modeldeki Açıklayıcı Değişken Sayısı')
ax.set_ylabel('R²')
ax.set_title('Rastgele Değişken Eklendikçe R² vs Düzeltilmiş R²')
ax.legend()
ax.axhline(y=r2_list[0], color='gray', linestyle=':', lw=1)
plt.tight_layout()
plt.show()
print('→ R² her zaman artar; Düzeltilmiş R² anlamsız değişken eklendiğinde düşer.')

---
<a id='5'></a>
## 5. Model Seçimi – Geriye Doğru Eleme

Sunumdaki **profesör değerlendirme** veri seti üzerinde model seçimi yapacağız.

**Strateji 1 – Düzeltilmiş R²:**
1. Tam modelle başla
2. Her değişkeni birer birer çıkar, Düz. R²'yi kaydet
3. Düz. R²'yi en çok artıran modeli seç
4. Tekrarla

**Strateji 2 – p-değeri:**
1. Tam modelle başla
2. En yüksek p-değerli değişkeni çıkar
3. Tüm değişkenler anlamlı olana kadar tekrarla

In [ ]:
# ── Profesör Değerlendirme Veri Seti ─────────────────────────────────────────
# openintro evals veri setine yakın sentetik veri
np.random.seed(42)
n = 463

guzellik      = np.random.normal(0, 0.7, n)
cinsiyet      = np.random.choice([0, 1], size=n, p=[0.42, 0.58])  # 0=kadın, 1=erkek
yas           = np.random.randint(28, 73, n)
resmi         = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
alt           = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
anadil        = np.random.choice([0, 1], size=n, p=[0.85, 0.15])  # 0=ingilizce, 1=değil
azinlik       = np.random.choice([0, 1], size=n, p=[0.85, 0.15])
ogrenci       = np.random.randint(8, 120, n)
kadro         = np.random.choice([0, 1, 2], size=n, p=[0.2, 0.3, 0.5])

# Gerçek model: sunumdaki katsayılar + gürültü
prof_deg = (4.63
            + 0.108 * guzellik
            + 0.204 * cinsiyet
            - 0.009 * yas
            + 0.151 * resmi
            + 0.058 * alt
            - 0.216 * anadil
            - 0.071 * azinlik
            - 0.0004 * ogrenci
            - 0.193 * (kadro == 1)
            - 0.157 * (kadro == 2)
            + np.random.normal(0, 0.45, n))
prof_deg = np.clip(prof_deg, 2.0, 5.0)

df_prof = pd.DataFrame({
    'degerlendirme': prof_deg,
    'guzellik': guzellik,
    'cinsiyet': cinsiyet,
    'yas': yas,
    'resmi': resmi,
    'alt': alt,
    'anadil': anadil,
    'azinlik': azinlik,
    'ogrenci': ogrenci,
    'kadro_aday': (kadro == 1).astype(int),
    'kadro_lu':   (kadro == 2).astype(int)
})

print('Veri seti boyutu:', df_prof.shape)
df_prof.head()

In [ ]:
# ── Tam Model ────────────────────────────────────────────────────────────────
tam_formul = ('degerlendirme ~ guzellik + cinsiyet + yas + resmi + alt'
              ' + anadil + azinlik + ogrenci + kadro_aday + kadro_lu')
tam_model = smf.ols(tam_formul, data=df_prof).fit()

print('=== TAM MODEL ===')
print(tam_model.summary().tables[1])
print(f'\nDüzeltilmiş R² = {tam_model.rsquared_adj:.4f}')

In [ ]:
# ── p-Değeri Yaklaşımıyla Geriye Doğru Eleme ─────────────────────────────────
def geri_eleme_p(df, bagimli, bagimsizlar, esik=0.05, verbose=True):
    """p-değeri yaklaşımıyla geriye doğru eleme."""
    mevcut = list(bagimsizlar)
    adim = 0
    while True:
        formul = bagimli + ' ~ ' + ' + '.join(mevcut)
        model = smf.ols(formul, data=df).fit()
        p_degerleri = model.pvalues.drop('Intercept')
        maks_p = p_degerleri.max()
        if maks_p > esik:
            cikar = p_degerleri.idxmax()
            if verbose:
                print(f'Adım {adim+1}: "{cikar}" çıkarıldı  (p={maks_p:.4f})  |  Kalan Düz.R²={model.rsquared_adj:.4f}')
            mevcut.remove(cikar)
            adim += 1
        else:
            break
    return model, mevcut

tum_degiskenler = ['guzellik','cinsiyet','yas','resmi','alt',
                   'anadil','azinlik','ogrenci','kadro_aday','kadro_lu']

print('=== p-DEĞERİ YAKLAŞIMIYLA GERİYE DOĞRU ELEME ===')
son_model_p, kalan_p = geri_eleme_p(df_prof, 'degerlendirme', tum_degiskenler)
print(f'\nSon Model: {" + ".join(kalan_p)}')
print(son_model_p.summary().tables[1])
print(f'Düzeltilmiş R² = {son_model_p.rsquared_adj:.4f}')

In [ ]:
# ── Düzeltilmiş R² Yaklaşımıyla Geriye Doğru Eleme ──────────────────────────
def geri_eleme_r2(df, bagimli, bagimsizlar, verbose=True):
    """Düzeltilmiş R² yaklaşımıyla geriye doğru eleme."""
    mevcut = list(bagimsizlar)
    formul = bagimli + ' ~ ' + ' + '.join(mevcut)
    mevcut_r2 = smf.ols(formul, data=df).fit().rsquared_adj
    adim = 0
    while len(mevcut) > 1:
        r2_tablosu = {}
        for deg in mevcut:
            deneme = [v for v in mevcut if v != deg]
            f = bagimli + ' ~ ' + ' + '.join(deneme)
            r2_tablosu[deg] = smf.ols(f, data=df).fit().rsquared_adj
        en_iyi_cikar = max(r2_tablosu, key=r2_tablosu.get)
        en_iyi_r2    = r2_tablosu[en_iyi_cikar]
        if en_iyi_r2 >= mevcut_r2:
            if verbose:
                print(f'Adım {adim+1}: "{en_iyi_cikar}" çıkarıldı  |  Yeni Düz.R²={en_iyi_r2:.4f}')
            mevcut.remove(en_iyi_cikar)
            mevcut_r2 = en_iyi_r2
            adim += 1
        else:
            break
    formul = bagimli + ' ~ ' + ' + '.join(mevcut)
    return smf.ols(formul, data=df).fit(), mevcut

print('=== DÜZELTİLMİŞ R² YAKLAŞIMIYLA GERİYE DOĞRU ELEME ===')
son_model_r2, kalan_r2 = geri_eleme_r2(df_prof, 'degerlendirme', tum_degiskenler)
print(f'\nSon Model: {" + ".join(kalan_r2)}')
print(f'Düzeltilmiş R² = {son_model_r2.rsquared_adj:.4f}')

---
<a id='6'></a>
## 6. Model Koşullarının Kontrolü

Çoklu doğrusal regresyon modeli 4 koşula dayanır:
1. **Normallik:** Artıklar yaklaşık normal dağılımlı
2. **Sabit varyans (homoskedastisite):** Artıkların varyansı tahmin edilen değerlerle değişmemeli
3. **Bağımsızlık:** Artıklar bağımsız
4. **Doğrusallık:** Her açıklayıcı değişken sonuçla doğrusal ilişkili

In [ ]:
# ── 4 Koşul Diagnostik Grafikleri ────────────────────────────────────────────
artiklar    = son_model_p.resid.values
uydurulmus  = son_model_p.fittedvalues.values
siralama    = np.arange(1, len(artiklar)+1)

fig, axs = plt.subplots(2, 2, figsize=(11, 8))
fig.suptitle('Model Koşullarının Grafik Kontrolü', fontsize=14, y=1.01)

# (1) Normallik: Artıkların Histogramı
axs[0,0].hist(artiklar, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axs[0,0].set_title('(1) Artıkların Histogramı\n→ Normal görünüyor?')
axs[0,0].set_xlabel('Artıklar')
axs[0,0].set_ylabel('Frekans')

# (1b) Normal Q-Q Grafiği
stats.probplot(artiklar, dist='norm', plot=axs[0,1])
axs[0,1].set_title('(1b) Normal Olasılık Grafiği (Q-Q)\n→ Noktalar diyagonal üzerinde mi?')
axs[0,1].get_lines()[0].set(color='steelblue', markersize=3)

# (2) Sabit Varyans: Artıklar vs Uydurulan Değerler
axs[1,0].scatter(uydurulmus, artiklar, alpha=0.5, s=18, color='steelblue')
axs[1,0].axhline(0, color='tomato', lw=1.5, linestyle='--')
axs[1,0].set_title('(2) Artıklar vs Uydurulan Değerler\n→ Rastgele dağılım (sabit varyans)?')
axs[1,0].set_xlabel('Uydurulan (Tahmin)')
axs[1,0].set_ylabel('Artıklar')

# (3) Bağımsızlık: Artıklar vs Veri Toplama Sırası
axs[1,1].scatter(siralama, artiklar, alpha=0.4, s=10, color='steelblue')
axs[1,1].axhline(0, color='tomato', lw=1.5, linestyle='--')
axs[1,1].set_title('(3) Artıklar vs Veri Toplama Sırası\n→ Eğilim yok mu?')
axs[1,1].set_xlabel('Gözlem Sırası')
axs[1,1].set_ylabel('Artıklar')

plt.tight_layout()
plt.show()

In [ ]:
# ── (4) Doğrusallık: Artıklar vs Sayısal Değişkenler ─────────────────────────
sayisal_deg = ['guzellik', 'yas']

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for i, deg in enumerate(sayisal_deg):
    axs[i].scatter(df_prof[deg], artiklar, alpha=0.4, s=12, color='steelblue')
    axs[i].axhline(0, color='tomato', lw=1.5, linestyle='--')
    axs[i].set_xlabel(deg)
    axs[i].set_ylabel('Artıklar')
    axs[i].set_title(f'(4) Artıklar vs {deg}\n→ Doğrusal ilişki?')
plt.suptitle('Doğrusallık Kontrolü', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='7'></a>
## 7. Lojistik Regresyon ve GLM

**İkili kategorik** bir yanıt değişkeni için kullanılır (örn: hayatta kaldı/öldü, spam/değil).

**Lojit fonksiyonu (bağlantı fonksiyonu):**
$$\text{logit}(p) = \log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

**Tersine çevirerek tahmin:**
$$p = \frac{e^{\eta}}{1 + e^{\eta}} = \frac{1}{1 + e^{-\eta}}$$

In [ ]:
# ── Lojit ve Ters Lojit Fonksiyonlarını Görselleştirme ────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(11, 4))

# Lojit: p → (-∞, +∞)
p_aralik = np.linspace(0.001, 0.999, 300)
logit_val = np.log(p_aralik / (1 - p_aralik))
axs[0].plot(p_aralik, logit_val, color='steelblue', lw=2.5)
axs[0].axhline(0, color='gray', lw=0.8, ls=':')
axs[0].axvline(0.5, color='tomato', lw=1, ls='--', label='p=0.5 → logit=0')
axs[0].set_xlabel('p (olasılık)')
axs[0].set_ylabel('logit(p)')
axs[0].set_title('Lojit Fonksiyonu\n[0,1] → (-∞, +∞)')
axs[0].legend()

# Ters lojit (lojistik/sigmoid): η → [0,1]
eta_aralik = np.linspace(-6, 6, 300)
sigmoid    = 1 / (1 + np.exp(-eta_aralik))
axs[1].plot(eta_aralik, sigmoid, color='steelblue', lw=2.5)
axs[1].axhline(0.5, color='tomato', lw=1, ls='--', label='p=0.5')
axs[1].axhline(0.0, color='gray', lw=0.8, ls=':')
axs[1].axhline(1.0, color='gray', lw=0.8, ls=':')
axs[1].set_xlabel('η (doğrusal tahmin)')
axs[1].set_ylabel('p = σ(η)')
axs[1].set_title('Ters Lojit (Sigmoid) Fonksiyonu\n(-∞, +∞) → [0,1]')
axs[1].legend()

plt.tight_layout()
plt.show()

print('Şans Oranı (Odds) Hatırlatıcı:')
for p in [0.1, 0.5, 0.8, 0.9]:
    odds = p / (1 - p)
    print(f'  p = {p:.1f}  →  odds = {odds:.3f}  →  log-odds = {np.log(odds):.3f}')

---
<a id='8'></a>
## 8. Örnek – Donner Partisi: Hayatta Kalma Analizi

1846-1847 Sierra Nevada felaketi. 45 kişilik veri seti. Yanıt değişkeni: `durum` (1=Kurtuldu, 0=Öldü).

In [ ]:
# ── Donner Partisi Verisi ─────────────────────────────────────────────────────
donner_data = [
    (23,'Erkek',0),(40,'Kadın',1),(40,'Erkek',1),(30,'Erkek',0),(28,'Erkek',0),
    (40,'Erkek',0),(45,'Kadın',0),(62,'Erkek',0),(65,'Erkek',0),(45,'Erkek',0),
    (25,'Kadın',1),(28,'Kadın',1),(28,'Erkek',0),(23,'Erkek',1),(22,'Erkek',0),
    (23,'Erkek',0),(28,'Erkek',0),(15,'Erkek',0),(47,'Erkek',0),(57,'Erkek',0),
    (20,'Kadın',1),(18,'Kadın',1),(25,'Erkek',0),(60,'Erkek',0),(25,'Erkek',1),
    (35,'Kadın',1),(32,'Erkek',1),(24,'Kadın',1),(23,'Erkek',0),(28,'Kadın',1),
    (15,'Erkek',0),(50,'Erkek',0),(21,'Kadın',1),(25,'Erkek',0),(46,'Erkek',1),
    (32,'Kadın',0),(30,'Erkek',1),(25,'Kadın',1),(25,'Erkek',0),(28,'Erkek',1),
    (32,'Erkek',0),(20,'Erkek',0),(23,'Erkek',1),(24,'Erkek',0),(25,'Kadın',1)
]
donner = pd.DataFrame(donner_data, columns=['yas', 'cinsiyet', 'durum'])
donner['cinsiyet_kodu'] = (donner['cinsiyet'] == 'Kadın').astype(int)

print(f'Toplam: {len(donner)} kişi')
print(donner.groupby(['cinsiyet','durum']).size().unstack().rename(columns={0:'Öldü',1:'Kurtuldu'}))

In [ ]:
# ── Model 1: Sadece Yaş ───────────────────────────────────────────────────────
model_yas = smf.logit('durum ~ yas', data=donner).fit()
print(model_yas.summary())

In [ ]:
# ── Sunumdaki Tahminlerin Doğrulanması ───────────────────────────────────────
b0_yas = model_yas.params['Intercept']
b1_yas = model_yas.params['yas']

def log_sansoru(yas_val):
    return b0_yas + b1_yas * yas_val

def hayatta_kalma_olasiligi(yas_val):
    eta = log_sansoru(yas_val)
    odds = np.exp(eta)
    p = odds / (1 + odds)
    return eta, odds, p

print('Sunumdaki Hesaplamalar:')
print(f'Katsayılar: Kesim={b0_yas:.4f}, Yaş={b1_yas:.4f}\n')
for yas_test in [0, 25, 50]:
    eta, odds, p = hayatta_kalma_olasiligi(yas_test)
    print(f'  Yaş={yas_test:2d}:  log-odds={eta:.4f}  →  odds={odds:.3f}  →  p={p:.3f}')

In [ ]:
# ── Model 2: Yaş + Cinsiyet ───────────────────────────────────────────────────
model_yas_cin = smf.logit('durum ~ yas + cinsiyet_kodu', data=donner).fit()
print(model_yas_cin.summary())

b0 = model_yas_cin.params['Intercept']
b_yas = model_yas_cin.params['yas']
b_cin = model_yas_cin.params['cinsiyet_kodu']
print(f'\nŞans Oranı Yorumu:')
print(f'  Yaştaki 1 yıllık artış için: exp({b_yas:.4f}) = {np.exp(b_yas):.4f}')
print(f'  Kadın vs Erkek (yaş sabit):  exp({b_cin:.4f}) = {np.exp(b_cin):.4f}')

In [ ]:
# ── Lojistik Eğri: Erkek ve Kadın Modelleri ──────────────────────────────────
yas_aralik = np.linspace(15, 70, 200)

p_erkek = 1 / (1 + np.exp(-(b0 + b_yas * yas_aralik)))
p_kadin = 1 / (1 + np.exp(-(b0 + b_cin + b_yas * yas_aralik)))

fig, ax = plt.subplots(figsize=(8, 5))

# Veri noktaları (jitter ile)
for _, row in donner.iterrows():
    jitter = np.random.uniform(-0.02, 0.02)
    color = 'tomato' if row['cinsiyet'] == 'Kadın' else 'steelblue'
    marker = 'D' if row['cinsiyet'] == 'Kadın' else 'o'
    ax.scatter(row['yas'], row['durum'] + jitter, color=color, marker=marker,
               s=40, alpha=0.7, zorder=3)

ax.plot(yas_aralik, p_erkek, color='steelblue', lw=2.5, label='Erkek')
ax.plot(yas_aralik, p_kadin, color='tomato',    lw=2.5, label='Kadın')

ax.set_xlabel('Yaş')
ax.set_ylabel('Hayatta Kalma Olasılığı')
ax.set_title('Donner Partisi – Hayatta Kalma Olasılığı (Yaş + Cinsiyet)')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Öldü (0)', 'Kurtuldu (1)'])
ax.legend()
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print('Yorum: Kadınların ve genç yaştakilerin hayatta kalma olasılığı daha yüksek.')

In [ ]:
# ── Güven Aralığı: Yaş Katsayısı ─────────────────────────────────────────────
GA_log_odds = (
    b_yas - 1.96 * model_yas_cin.bse['yas'],
    b_yas + 1.96 * model_yas_cin.bse['yas']
)
GA_odds = (np.exp(GA_log_odds[0]), np.exp(GA_log_odds[1]))

print('Yaş katsayısı için %95 Güven Aralığı:')
print(f'  Log-odds GA: ({GA_log_odds[0]:.4f}, {GA_log_odds[1]:.4f})')
print(f'  Şans Oranı GA: ({GA_odds[0]:.4f}, {GA_odds[1]:.4f})')
print(f'  Yorum: 1 yaş büyük her kişinin hayatta kalma şans oranı {GA_odds[0]:.3f}–{GA_odds[1]:.3f} kat.')

---
<a id='9'></a>
## 9. Örnek – Kuş Beslemek ve Akciğer Kanseri

Hollanda vaka-kontrol çalışması (1985). Yanıt: akciğer kanseri var/yok.

In [ ]:
# ── Sentetik Veri (Sunumdaki Katsayılara Göre) ───────────────────────────────
np.random.seed(21)
n_kus = 147

cinsiyet_k = np.random.choice([0, 1], n_kus, p=[0.5, 0.5])   # 0=Erkek, 1=Kadın
se_k       = np.random.choice([0, 1], n_kus, p=[0.5, 0.5])   # 0=Düşük, 1=Yüksek
kus_k      = np.random.choice([0, 1], n_kus, p=[0.45, 0.55]) # 0=KuşYok, 1=Kuş
yas_k      = np.random.randint(30, 66, n_kus)
iy_k       = np.random.randint(0, 40, n_kus)
gs_k       = np.random.randint(0, 30, n_kus)

# Gerçek model: sunumdaki katsayılar
eta_k = (-1.937
         + 0.561 * cinsiyet_k
         + 0.105 * se_k
         + 1.363 * kus_k
         - 0.040 * yas_k
         + 0.073 * iy_k
         + 0.026 * gs_k)
p_k = 1 / (1 + np.exp(-eta_k))
ak_k = np.random.binomial(1, p_k)

df_kus = pd.DataFrame({
    'AK': ak_k, 'CE': cinsiyet_k, 'SE': se_k,
    'KB': kus_k, 'YA': yas_k, 'IY': iy_k, 'GS': gs_k
})
print(f'Veri: {n_kus} kişi | Akciğer Kanseri: {ak_k.sum()} | Kanser Yok: {(ak_k==0).sum()}')

In [ ]:
# ── Lojistik Regresyon Modeli ─────────────────────────────────────────────────
model_kus = smf.logit('AK ~ CE + SE + KB + YA + IY + GS', data=df_kus).fit()
print(model_kus.summary())

In [ ]:
# ── Şans Oranı Yorumu ─────────────────────────────────────────────────────────
katsayilar = model_kus.params
print('Şans Oranları (exp(β)):')
so_df = pd.DataFrame({
    'Katsayı (β)':  katsayilar.round(4),
    'Şans Oranı': np.exp(katsayilar).round(4)
})
print(so_df.to_string())

b_kus_val = katsayilar['KB']
b_iy_val  = katsayilar['IY']
print(f'\nAnahtar Yorumlar:')
print(f'  Kuş besleme ŞO: exp({b_kus_val:.4f}) = {np.exp(b_kus_val):.2f}')
print(f'  → Kuş besleyenler, kuş beslemeyenlere kıyasla {np.exp(b_kus_val):.2f}x daha fazla akciğer kanseri riskine sahip.')
print(f'  Sigara (1 yıl) ŞO: exp({b_iy_val:.4f}) = {np.exp(b_iy_val):.3f}')
print(f'  → Her ek sigara yılı için risk {np.exp(b_iy_val):.3f}x artıyor.')

In [ ]:
# ── Şans Oranı ≠ Göreli Risk ─────────────────────────────────────────────────
# Sunumdaki hesaplama: P(kanser|kuş yok) = 0.05 varsayımı altında
so_kus = np.exp(b_kus_val)  # tahmin edilen ŞO
p_kus_yok = 0.05

p_kus_var = (so_kus * (p_kus_yok / (1 - p_kus_yok))) / (
             1 + so_kus * (p_kus_yok / (1 - p_kus_yok)))
gorel_risk = p_kus_var / p_kus_yok

print('Şans Oranı vs Göreli Risk (Relative Risk):')
print(f'  Varsayım: P(kanser|kuş yok) = {p_kus_yok}')
print(f'  P(kanser|kuş var)           = {p_kus_var:.3f}')
print(f'  Göreli Risk (GR)            = {gorel_risk:.2f}')
print(f'  Şans Oranı (ŞO)             = {so_kus:.2f}')
print(f'\n  ⚠  GR={gorel_risk:.2f} ≠ ŞO={so_kus:.2f}')
print(f'  → Lojistik regresyon "4 kat daha olası" demez, "şans oranı {so_kus:.2f}x" der.')

---
<a id='10'></a>
## 10. ROC Eğrisi, AUC ve Spam Filtresi

### Temel Kavramlar

| Gerçek \ Tahmin | Pozitif | Negatif |
|---|---|---|
| **Pozitif** | Gerçek Pozitif (GP) | Yanlış Negatif (YN) |
| **Negatif** | Yanlış Pozitif (YP) | Gerçek Negatif (GN) |

$$\text{Duyarlılık} = \frac{GP}{GP+YN} \qquad \text{Özgüllük} = \frac{GN}{YP+GN}$$

In [ ]:
# ── Spam Veri Seti (Basitleştirilmiş) ────────────────────────────────────────
np.random.seed(99)
n_spam = 3921
spam_gercek = np.random.choice([0, 1], n_spam, p=[0.9, 0.1])

# Lojistik regresyonla tahmin edilen olasılıklar
p_tahmin_spam = np.where(
    spam_gercek == 1,
    np.random.beta(3, 2, n_spam),    # spam için yüksek olasılık
    np.random.beta(1, 5, n_spam)     # spam değil için düşük olasılık
)

print(f'Toplam e-posta: {n_spam}')
print(f'Spam:           {spam_gercek.sum()} (%{spam_gercek.mean()*100:.1f})')
print(f'Spam Değil:     {(spam_gercek==0).sum()} (%{(spam_gercek==0).mean()*100:.1f})')

In [ ]:
# ── Farklı Eşikler için Duyarlılık / Özgüllük ────────────────────────────────
esikler = [0.75, 0.625, 0.5, 0.375, 0.25]

print(f'{"Eşik":>8} {"GP":>6} {"GN":>6} {"YP":>6} {"YN":>6} {"Duyarlılık":>12} {"Özgüllük":>10}')
print('-' * 65)
for esik in esikler:
    tahmin = (p_tahmin_spam >= esik).astype(int)
    cm = confusion_matrix(spam_gercek, tahmin)
    GN, YP, YN, GP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    duyarlilik = GP / (GP + YN) if (GP + YN) > 0 else 0
    ozgulluk   = GN / (YP + GN) if (YP + GN) > 0 else 0
    print(f'{esik:>8.3f} {GP:>6} {GN:>6} {YP:>6} {YN:>6} {duyarlilik:>12.3f} {ozgulluk:>10.3f}')

In [ ]:
# ── ROC Eğrisi ve AUC ────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(spam_gercek, p_tahmin_spam)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2.5, label=f'ROC Eğrisi (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Rastgele Sınıflandırıcı (AUC = 0.5)')

# Sunumdaki eşikleri işaretle
for esik_val in [0.75, 0.5, 0.25]:
    idx = np.argmin(np.abs(thresholds - esik_val))
    ax.plot(fpr[idx], tpr[idx], 'ro', markersize=8)
    ax.annotate(f'p={esik_val}', (fpr[idx]+0.02, tpr[idx]-0.04), fontsize=8, color='tomato')

ax.set_xlabel('Yanlış Pozitif Oranı (1 – Özgüllük)')
ax.set_ylabel('Doğru Pozitif Oranı (Duyarlılık)')
ax.set_title('ROC Eğrisi – Spam Filtresi')
ax.legend(loc='lower right')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

print(f'AUC = {roc_auc:.3f}')
print('→ AUC=1 mükemmel model, AUC=0.5 rastgele tahmin.')
print('→ ROC eğrisi tüm olası eşikler için duyarlılık-özgüllük dengesini gösterir.')

In [ ]:
# ── Fayda Fonksiyonu ile Optimum Eşik ────────────────────────────────────────
# Sunumdaki fayda: GP=+1, GN=+1, YP=-50, YN=-5
def fayda(y_gercek, y_tahmin):
    cm = confusion_matrix(y_gercek, y_tahmin)
    GN, YP, YN, GP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    return GP * 1 + GN * 1 + YP * (-50) + YN * (-5)

esik_aralik = np.linspace(0.01, 0.99, 200)
fayda_degerleri = []

for e in esik_aralik:
    tahmin_e = (p_tahmin_spam >= e).astype(int)
    fayda_degerleri.append(fayda(spam_gercek, tahmin_e))

en_iyi_idx  = np.argmax(fayda_degerleri)
en_iyi_esik = esik_aralik[en_iyi_idx]
en_iyi_fayda = fayda_degerleri[en_iyi_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(esik_aralik, fayda_degerleri, color='steelblue', lw=2)
ax.axvline(en_iyi_esik, color='tomato', lw=1.5, linestyle='--',
            label=f'Optimum eşik = {en_iyi_esik:.2f}')
ax.scatter([en_iyi_esik], [en_iyi_fayda], color='tomato', s=80, zorder=5)
ax.set_xlabel('Eşik (p)')
ax.set_ylabel('Fayda F(p)')
ax.set_title('Fayda Fonksiyonu – Optimum Eşik Seçimi')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Optimum Eşik: {en_iyi_esik:.3f}')
print(f'Maksimum Fayda: {en_iyi_fayda}')
print(f'0.75 Eşiği Faydası: {fayda(spam_gercek, (p_tahmin_spam>=0.75).astype(int))}')

In [ ]:
# ── Duyarlılık ve Özgüllük – Denge Görselleştirmesi ──────────────────────────
duyarlilik_list = []
ozgulluk_list   = []

for e in esik_aralik:
    tahmin_e = (p_tahmin_spam >= e).astype(int)
    cm = confusion_matrix(spam_gercek, tahmin_e)
    GN, YP, YN, GP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    duyarlilik_list.append(GP / (GP+YN) if (GP+YN)>0 else 0)
    ozgulluk_list.append(GN / (YP+GN) if (YP+GN)>0 else 0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(esik_aralik, duyarlilik_list, lw=2, color='steelblue', label='Duyarlılık')
ax.plot(esik_aralik, ozgulluk_list,   lw=2, color='tomato',    label='Özgüllük')
ax.axvline(en_iyi_esik, color='gray', lw=1.5, linestyle='--', label=f'Optimum eşik={en_iyi_esik:.2f}')
ax.set_xlabel('Eşik (p)')
ax.set_ylabel('Oran')
ax.set_title('Eşiğe Göre Duyarlılık ve Özgüllük Dengesi')
ax.legend()
plt.tight_layout()
plt.show()

print('→ Eşik azaldığında: Duyarlılık artar, Özgüllük azalır.')
print('→ Eşik arttığında:  Duyarlılık azalır, Özgüllük artar.')
print('→ Optimum eşik, fayda fonksiyonuna göre belirlenir.')

---
## 📌 Bölüm Özeti

| Konu | Anahtar Nokta |
|---|---|
| **Çoklu Regresyon** | $\hat{y} = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$ |
| **Kategorik Değişken** | $k-1$ kukla değişken; biri referans düzeyi |
| **Katsayı Yorumu** | "Diğer her şey sabit tutulduğunda..." |
| **R² vs Düzeltilmiş R²** | Düzeltilmiş R² fazladan değişkene ceza uygular |
| **Model Seçimi** | Geriye doğru eleme: p-değeri veya Düzeltilmiş R² ile |
| **Çoklu Doğrusal Bağlantı** | VIF > 5–10 → sorun; birbiriyle ilişkili değişkenleri birlikte modele alma |
| **Model Koşulları** | Normallik, Sabit Varyans, Bağımsızlık, Doğrusallık |
| **Lojistik Regresyon** | İkili yanıt; logit bağlantı fonksiyonu; şans oranı yorumu |
| **ROC / AUC** | Tüm eşikler için duyarlılık-özgüllük dengesi |
| **ŞO ≠ Göreli Risk** | Düşük prevalansta büyük fark yaratır |

